<a href="https://colab.research.google.com/github/hyunkyung31/Dajeon_JS/blob/main/hyunkyung/RT_DETR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1) bbox → YOLO 포맷 변환 (탐지 공통 준비)     ← 가장 먼저
2) RT-DETR 학습 + test mAP                    ← 메인
3) Swin-T 분류 (frame_df + label)             ← 그다음

Step 1: bbox 있는 keyframe 목록 만들기 (탐지용 데이터 필터링)<br>
Step 2: CADICA bbox → YOLO 포맷([cx,cy,w,h] 정규화) 변환<br>
Step 3: images/labels 폴더 구조 + data.yaml 만들기<br>
Step 4: Ultralytics 설치 + RT-DETR 모델 불러오기<br>
Step 5: 소규모 학습 (몇 epoch 테스트)<br>
Step 6: 결과 확인 (mAP, 예측 bbox 시각화)<br>

 Ultralytics 라이브러리

분류(Swin-T)는 selectedFrames.txt에 있는 keyframe을 다 썼지만, 탐지는 bbox가 있는 프레임만 써야 <br>

groundtruth 폴더 안의 .txt 파일 하나하나가 "bbox 있는 프레임"

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/🐥new_2_project/04_DL_project/02_dataset/CADICA.zip'
extract_path = '/content/cadica_dataset/'

if os.path.exists(zip_path) :
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f'압축 해제 완료, 저장위치 : {extract_path}')
else :
    print(f'에러 {zip_path}에 파일이 존재하지 않습니다')

압축 해제 완료, 저장위치 : /content/cadica_dataset/


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/08_전처리/common_split.csv"
df = pd.read_csv(csv_path)

print(df.shape)
df.head()

(334, 7)


,patient_id,video_id,label,view,n_frames,has_groundtruth,split
0,p1,v2,lesion,LCA,56,True,train
1,p1,v3,lesion,LCA,44,True,train
2,p1,v10,lesion,LCA,56,True,train
3,p1,v11,lesion,LCA,44,True,train
4,p1,v1,non-lesion,LCA,55,False,train


In [ ]:
# @title Step 1 — bbox 있는 keyframe 목록 만들기

from pathlib import Path
import pandas as pd

base = Path("/content/cadica_dataset/CADICA/selectedVideos")

rows = []
for _, r in df.iterrows():
    gt_dir = base / r.patient_id / r.video_id / "groundtruth"
    if not gt_dir.exists():
        continue  # non-lesion 영상은 groundtruth 폴더 자체가 없음

    for gt_file in sorted(gt_dir.glob("*.txt")):
        if "groundTruthTable" in gt_file.name:  # .mat 관련 텍스트 파일 아님, 확인용
            continue
        frame_id = gt_file.stem  # 예: p1_v2_00015
        img_path = base / r.patient_id / r.video_id / "input" / f"{frame_id}.png"

        rows.append({
            "patient_id": r.patient_id,
            "video_id": r.video_id,
            "frame_id": frame_id,
            "img_path": str(img_path),
            "bbox_path": str(gt_file),
            "split": r.split,
            "view": r.view,
        })

det_df = pd.DataFrame(rows)
print("bbox 있는 프레임 수:", len(det_df))
print(det_df.split.value_counts())
det_df.head()

bbox 있는 프레임 수: 3685
split
train    3077
val       335
test      273
Name: count, dtype: int64


,patient_id,video_id,frame_id,img_path,bbox_path,split,view
0,p1,v2,p1_v2_00015,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,<bound method Series.view of patient_id ...
1,p1,v2,p1_v2_00016,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,<bound method Series.view of patient_id ...
2,p1,v2,p1_v2_00017,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,<bound method Series.view of patient_id ...
3,p1,v2,p1_v2_00018,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,<bound method Series.view of patient_id ...
4,p1,v2,p1_v2_00019,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,<bound method Series.view of patient_id ...


In [ ]:
det_df["img_exists"] = det_df.img_path.apply(lambda p: Path(p).exists())
det_df["bbox_exists"] = det_df.bbox_path.apply(lambda p: Path(p).exists())
print(det_df.img_exists.value_counts())
print(det_df.bbox_exists.value_counts())

img_exists
True    3685
Name: count, dtype: int64
bbox_exists
True    3685
Name: count, dtype: int64


CADICA groundtruth .txt  
x y w h category

(x, y) = 좌상단 픽셀 좌표<br>
(w, h) = 픽셀 단위 너비/높이<br>
category = p0_20 병변 등급<br>

YOLO/RT-DETR은<br>
class_id cx cy w h<br>
cx, cy, w, h 는 이미지 크기로 나눈 0~1값 <br>
cx, cy는 중심점 <br>



In [ ]:
sample_row = det_df.iloc[0]
print(sample_row.bbox_path)
print(Path(sample_row.bbox_path).read_text())

/content/cadica_dataset/CADICA/selectedVideos/p1/v2/groundtruth/p1_v2_00015.txt
257 123 42 55 p0_20



In [ ]:
import cv2
sample_img = cv2.imread(sample_row.img_path)
print('이미지 크기 (H, W, C):', sample_img.shape)

이미지 크기 (H, W, C): (512, 512, 3)


In [ ]:
# @title bbox 파싱 함수
def parse_bbox_file(bbox_path):
    boxes = []
    with open(bbox_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            x, y, w, h = map(int, parts[:4])
            category = parts[4] if len(parts) > 4 else "unknown"
            boxes.append({"x": x, "y": y, "w": w, "h": h, "category": category})
    return boxes

# 테스트
boxes = parse_bbox_file(sample_row.bbox_path)
print(boxes)

[{'x': 257, 'y': 123, 'w': 42, 'h': 55, 'category': 'p0_20'}]


In [ ]:
# @title CADICA -> YOLO 좌표 변환 함수

def cadica_to_yolo(x, y, w, h, img_w, img_h):
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    w_norm = w / img_w
    h_norm = h / img_h
    return cx, cy, w_norm, h_norm

# 테스트 (class_id는 지금은 전부 0 = "lesion" 하나로 통일)
img_h, img_w = sample_img.shape[:2]
for b in boxes:
    cx, cy, w_norm, h_norm = cadica_to_yolo(b["x"], b["y"], b["w"], b["h"], img_w, img_h)
    print(f"0 {cx:.6f} {cy:.6f} {w_norm:.6f} {h_norm:.6f}   (원본 category: {b['category']})")

0 0.542969 0.293945 0.082031 0.107422   (원본 category: p0_20)


Ultralytics(YOLOv8, RT-DETR 둘 다 같은 라이브러리)는 학습할 때 이 구조

cadica_yolo/<br>
├── images/<br>
│   ├── train/  (.png 파일들)<br>
│   ├── val/<br>
│   └── test/<br>
├── labels/<br>
│   ├── train/  (.txt 파일들, 이미지와 같은 이름)<br>
│   ├── val/<br>
│   └── test/<br>
└── data.yaml<br>

이미지와 라벨을 같은 파일명(확장자만 다름)으로, 같은 split 폴더에 넣어두면 라이브러리가 자동으로 짝을 맞춰 읽음

In [ ]:
# @title 폴더 만들기
import os

YOLO_ROOT = Path("/content/cadica_yolo")

for split in ["train", "val", "test"]:
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

print("폴더 생성 완료")
print(list(YOLO_ROOT.rglob("*")))

폴더 생성 완료
[PosixPath('/content/cadica_yolo/labels'), PosixPath('/content/cadica_yolo/images'), PosixPath('/content/cadica_yolo/labels/train'), PosixPath('/content/cadica_yolo/labels/test'), PosixPath('/content/cadica_yolo/labels/val'), PosixPath('/content/cadica_yolo/images/train'), PosixPath('/content/cadica_yolo/images/test'), PosixPath('/content/cadica_yolo/images/val')]


In [ ]:
# @title 이미지 크기를 매번 읽지 않고 캐싱 (속도 최적화)
# 파일별로 직접 확인, cv2.imread 대신 헤더만 읽는 방법

from PIL import Image

def get_image_size_fast(img_path):
    with Image.open(img_path) as im:
        return im.width, im.height  # (W, H)

In [ ]:
# @title  전체 변환 + symlink 생성 함수

import shutil

def process_one_frame(row):
    img_path = Path(row.img_path)
    bbox_path = Path(row.bbox_path)
    split = row.split
    frame_id = row.frame_id

    img_w, img_h = get_image_size_fast(img_path)
    boxes = parse_bbox_file(bbox_path)

    # 라벨(.txt) 생성
    label_lines = []
    for b in boxes:
        cx, cy, w_norm, h_norm = cadica_to_yolo(b["x"], b["y"], b["w"], b["h"], img_w, img_h)
        label_lines.append(f"0 {cx:.6f} {cy:.6f} {w_norm:.6f} {h_norm:.6f}")

    label_out_path = YOLO_ROOT / "labels" / split / f"{frame_id}.txt"
    label_out_path.write_text("\n".join(label_lines))

    # 이미지 symlink 생성
    img_out_path = YOLO_ROOT / "images" / split / f"{frame_id}.png"
    if not img_out_path.exists():
        os.symlink(img_path.resolve(), img_out_path)

    return len(boxes)

In [ ]:
# @title 전체 실행
from tqdm import tqdm

n_boxes_total = 0
errors = []

for _, row in tqdm(det_df.iterrows(), total=len(det_df)):
    try:
        n_boxes = process_one_frame(row)
        n_boxes_total += n_boxes
    except Exception as e:
        errors.append((row.frame_id, str(e)))

print("총 처리된 프레임 수:", len(det_df) - len(errors))
print("총 bbox 수:", n_boxes_total)
print("에러 개수:", len(errors))
if errors:
    print(errors[:5])

100%|██████████| 3685/3685 [00:02<00:00, 1823.62it/s]

총 처리된 프레임 수: 3685
총 bbox 수: 5835
에러 개수: 0


# data.yaml

Ultralytics(RT-DETR, YOLOv8 공통)가 "데이터가 어디 있고, 클래스가 몇 개고, 이름이 뭔지" 알아야 하는 설정

In [ ]:
# @title data.yaml 작성
yaml_content = f"""path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['lesion']
"""

yaml_path = YOLO_ROOT / "data.yaml"
yaml_path.write_text(yaml_content)

print(yaml_path.read_text())

path: /content/cadica_yolo
train: images/train
val: images/val
test: images/test

nc: 1
names: ['lesion']



In [ ]:
# @title 폴더에 실제 파일들이 잘 들어갔는지 확인
for split in ["train", "val", "test"]:
    n_img = len(list((YOLO_ROOT / "images" / split).glob("*.png")))
    n_lbl = len(list((YOLO_ROOT / "labels" / split).glob("*.txt")))
    print(f"{split}: images={n_img}, labels={n_lbl}")

train: images=3077, labels=3077
val: images=335, labels=335
test: images=273, labels=273


In [ ]:
# @title Ultralytics 설치
!pip install -q ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.4 MB/s eta 0:00:00


In [ ]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 52.6/235.7 GB disk)


Ultralytics는 내부적으로 전부 PyTorch(torch.nn.Module)로 구현

 RT-DETR 모델 불러와서 소규모 학습
 => 일단 소규모로 돌려보고 파이프라인 자체가 문제 없이 돌아가는지 확인하기 위해서 한 에프크 3정도로 실행해보기

In [ ]:
from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")  # Ultralytics에서 제공하는 사전학습 가중치
print(model.info())

# rtdetr-l => RT-DETR-Large, COCO로 사전학습된 가중치
# rtdetr-x.pt 가 더 큰 버전

rt-detr-l summary: 449 layers, 32,970,476 parameters, 0 gradients, 108.3 GFLOPs
(449, 32970476, 0, 108.3437056)


In [ ]:
result = model.train(
    data = str(yaml_path),
    epochs = 3,
    imgsz = 640,
    batch = 8,
    device = 0,
    project = '/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result',
    name = 'test_run',
)

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=test_run, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/3      6.56G      1.054      1.168     0.3228          6        640: 100% ━━━━━━━━━━━━ 385/385 1.4it/s 4:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.7it/s 12.7s
                   all        335        506     0.0966       0.13     0.0369     0.0103

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        2/3      6.96G     0.7508     0.7425     0.1893         25        640: 0% ──────────── 0/385  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        2/3      6.96G     0.7548     0.6887      0.199          7        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:12
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 3.0it/s 7.1s
                   all        335        506      0.116      0.083     0.0255    0.00941

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/3      7.01G      1.092     0.6323     0.3202         12        640: 0% ──────────── 0/385  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        3/3      7.01G     0.6895     0.6827     0.1782         15        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:12
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.3s
                   all        335        506      0.159        0.2      0.077     0.0223

3 epochs completed in 0.226 hours.
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/test_run/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/test_run/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/test_run/weights/best.pt...
Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/2

Epoch 1/3
전체 3번 중 현재 1번째 <br>
GPU_mem 6.39G
지금 GPU 메모리 6.39GB 사용 중 (T4의 14.9GB 중, 여유 있음)<br>

giou_loss
bbox 위치·크기가 얼마나 정확한지에 대한 손실 (Generalized IoU 기반) — 낮을수록 좋음<br>

cls_loss
bbox 안의 클래스(lesion) 분류가 맞는지에 대한 손실 — 낮을수록 좋음<br>

l1_loss
bbox 좌표의 세부 오차 (또 다른 위치 손실) — 낮을수록 좋음<br>

Instances 35
이번 배치에 있던 bbox 개수<br>

Size 640
입력 이미지 크기

In [ ]:
from ultralytics import RTDETR

model = RTDETR("/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/test_run/weights/best.pt")

results = model.train(
    data=str(yaml_path),
    epochs=50,
    patience=20,       # val mAP 20 epoch 개선 없으면 자동 종료
    imgsz=640,
    batch=8,
    device=0,
    cache=True,         # 2 epoch부터 이미지 로딩 빨라짐
    project="/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result",
    name="train_50ep",
)

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/test_run/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=trai

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      7.33G     0.6457     0.6056     0.1625          6        640: 100% ━━━━━━━━━━━━ 385/385 1.4it/s 4:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.2s
                   all        335        506      0.104      0.166     0.0462     0.0116

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      6.81G     0.6519     0.5663     0.1618         25        640: 0% ──────────── 0/385  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      6.81G     0.6669      0.643     0.1721          7        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.8it/s 7.5s
                   all        335        506      0.114        0.2     0.0595     0.0179

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      6.86G       0.83     0.6571     0.1703         12        640: 0% ──────────── 0/385  0.7s

...

      28/50      6.86G     0.4792     0.4554      0.113         15        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:16
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.2s
                   all        335        506      0.144       0.18      0.061     0.0153

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      29/50      6.86G     0.3969     0.3913    0.07656         22        640: 0% ──────────── 0/385  0.8s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/50      6.86G     0.4724     0.4513     0.1109          5        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.3s
                   all        335        506        0.2      0.152     0.0698     0.0214

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/50      6.86G     0.5223     0.5034     0.1528         19        640: 0% ──────────── 0/385  0.8s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/50      6.86G     0.4659     0.4529     0.1085         11        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.3s
                   all        335        506      0.196      0.101     0.0497      0.016
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 10, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

30 epochs completed in 2.310 hours.
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_50ep-2/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_50ep-2/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_50ep-2/wei

EarlyStopping: 최근 20 epoch 동안 개선 없음 → epoch 10에서 멈춤<br>
Best model: epoch 10, best.pt<br>
최종 val 성능 (best.pt):<br>
  Precision: 0.208<br>
  Recall:    0.126<br>
  mAP50:     0.108<br>
  mAP50-95:  0.039<br>
총 30 epoch, 2.31시간 소요 (계획한 50 epoch까지 안 가고 자동 멈춤)<br>
best는 epoch 10 — 그 뒤 20 epoch(11~30) 동안 mAP가 이걸 못 넘겼음<br>

loss는 계속 내려가서 학습 자체는 진행되는데, mAP는 실질적인 개선 없음
Recall 0.126으로 병변의 약 87%를 놓침 (임상적으로 부족)
mAP50-95 0.039로 박스 정교함도 낮음

In [ ]:
import numpy as np

bad = []
for _, row in det_df.iterrows():
    boxes = parse_bbox_file(row.bbox_path)
    for b in boxes:
        if b["w"] <= 0 or b["h"] <= 0:
            bad.append((row.frame_id, b))

print("비정상 bbox 개수:", len(bad))
print(bad[:5])

비정상 bbox 개수: 0
[]


In [ ]:
from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")  # 이전 체크포인트 아니라 원본 COCO pretrained

results = model.train(
    data=str(yaml_path),
    epochs=50,
    patience=20,
    imgsz=640,
    batch=8,
    device=0,
    cache=True,
    lr0=0.0005,
    optimizer="AdamW",
    amp=False,          # ★ 추가 — mixed precision 끄기 (NaN 방지)
    project="/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result",
    name="train_v3_nofp16",
)

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_v3_nofp16-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      9.86G      1.366     0.5702     0.4461          6        640: 100% ━━━━━━━━━━━━ 385/385 1.3s/it 8:26
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.4it/s 15.2s
                   all        335        506    0.00331      0.259    0.00171   0.000419

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      9.96G     0.9371      0.897     0.2334          7        640: 100% ━━━━━━━━━━━━ 385/385 1.3s/it 8:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.4it/s 15.0s
                   all        335        506     0.0177      0.265    0.00955     0.0023

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       6/50      10.1G     0.5975     0.9762     0.1415          9        640: 100% ━━━━━━━━━━━━ 385/385 1.3s/it 8:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.4it/s 15.2s
                   all        335        506     0.0853      0.182      0.035     0.0103

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       7/50      10.1G     0.6363     0.8807     0.1455         21        640: 4% ──────────── 16/385 1.6it/s 21.5s<3:49


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
w = Path("/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_nofp16-2/weights")
# 폴더명이 다르면 실제 name으로 바꾸세요
print(list(w.glob("*.pt")) if w.exists() else "경로 확인 필요")

[PosixPath('/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_nofp16-2/weights/last.pt'), PosixPath('/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_nofp16-2/weights/best.pt')]


In [ ]:
from ultralytics import RTDETR

model = RTDETR("/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_nofp16-2/weights/last.pt")
# ↑ 실제 경로로 수정

results = model.train(
    data=str(yaml_path),
    epochs=50,
    patience=20,
    imgsz=640,
    batch=8,
    device=0,
    cache=True,
    lr0=0.0005,
    optimizer="AdamW",
    amp=True,            # ★ 속도 회복
    project="/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result",
    name="train_v3_amp_on",
)

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_nofp16-2/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      10.1G     0.8647     0.7768     0.2112          6        640: 100% ━━━━━━━━━━━━ 385/385 1.4it/s 4:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.3s
                   all        335        506      0.111      0.117     0.0232    0.00609

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      6.64G     0.9896     0.6455     0.1643         25        640: 0% ──────────── 0/385  2.0s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      6.64G     0.7535     0.7027     0.1804          7        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.2s
                   all        335        506      0.203      0.237     0.0893     0.0223

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      6.69G     0.9696     0.4744     0.2082         12        640: 0% ──────────── 0/385  1.8s

...

      29/50      6.69G     0.4725     0.4538     0.1084          5        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.2s
                   all        335        506      0.232      0.186     0.0809     0.0173

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/50      6.69G     0.3963      0.445     0.1116         19        640: 0% ──────────── 0/385  1.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/50      6.69G     0.4506     0.4486     0.1034         11        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:25
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 3.0it/s 7.0s
                   all        335        506      0.134       0.16     0.0469      0.011

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      31/50      6.69G     0.4511     0.4515     0.1056         10        640: 100% ━━━━━━━━━━━━ 385/385 1.4it/s 4:26
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.2s
                   all        335        506      0.194      0.204     0.0771      0.018
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 11, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

31 epochs completed in 2.741 hours.
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_amp_on/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_amp_on/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_a

- EarlyStopping @ epoch 11 (patience=20, 31 epoch까지 돌고 종료)
- Best (ep11): P=0.228, R=0.225, mAP50=0.118, mAP50-95=0.0305
- 이전 런(ep10, mAP50=0.108)보다 살짝 개선 → 현재까지 best.pt 기준

In [ ]:
# @title best.pt로 val + test 평가
from ultralytics import RTDETR

BEST = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_amp_on/weights/best.pt"
model = RTDETR(BEST)

val = model.val(data="/content/cadica_yolo/data.yaml", split="val", imgsz=640, batch=8)
test = model.val(data="/content/cadica_yolo/data.yaml", split="test", imgsz=640, batch=8)

print(f"VAL  P={val.box.mp:.3f} R={val.box.mr:.3f} mAP50={val.box.map50:.3f} mAP50-95={val.box.map:.3f}")
print(f"TEST P={test.box.mp:.3f} R={test.box.mr:.3f} mAP50={test.box.map50:.3f} mAP50-95={test.box.map:.3f}")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2424.9±632.3 MB/s, size: 95.5 KB)
val: Scanning /content/cadica_yolo/labels/val... 335 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 335/335 1.6Kit/s 0.2s
val: New cache created: /content/cadica_yolo/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 42/42 2.9it/s 14.5s
                   all        335        506      0.229      0.219      0.117       0.03
Speed: 1.5ms preprocess, 33.8ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2223.2±12

In [ ]:
# @title qualitative
model.predict(
    source="/content/cadica_yolo/images/test",
    conf=0.25, imgsz=640, save=True,
    project="/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result",
    name="predict_test_v3"
)


image 1/273 /content/cadica_yolo/images/test/p28_v6_00013.png: 640x640 1 lesion, 72.3ms
image 2/273 /content/cadica_yolo/images/test/p28_v6_00014.png: 640x640 2 lesions, 69.7ms
image 3/273 /content/cadica_yolo/images/test/p28_v6_00015.png: 640x640 3 lesions, 70.3ms
image 4/273 /content/cadica_yolo/images/test/p28_v6_00020.png: 640x640 1 lesion, 69.4ms
image 5/273 /content/cadica_yolo/images/test/p28_v6_00021.png: 640x640 1 lesion, 55.9ms
image 6/273 /content/cadica_yolo/images/test/p28_v6_00022.png: 640x640 2 lesions, 55.4ms
image 7/273 /content/cadica_yolo/images/test/p28_v6_00025.png: 640x640 2 lesions, 54.2ms
image 8/273 /content/cadica_yolo/images/test/p28_v6_00028.png: 640x640 1 lesion, 56.0ms
image 9/273 /content/cadica_yolo/images/test/p28_v6_00029.png: 640x640 2 lesions, 55.5ms
image 10/273 /content/cadica_yolo/images/test/p28_v6_00030.png: 640x640 1 lesion, 51.4ms
image 11/273 /content/cadica_yolo/images/test/p28_v7_00023.png: 640x640 1 lesion, 52.3ms
image 12/273 /content/ca

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'lesion'}
 obb: None
 orig_img: array([[[32, 32, 32],
         [30, 30, 30],
         [30, 30, 30],
         ...,
         [13, 13, 13],
         [15, 15, 15],
         [18, 18, 18]],
 
        [[30, 30, 30],
         [32, 32, 32],
         [34, 34, 34],
         ...,
         [17, 17, 17],
         [17, 17, 17],
         [18, 18, 18]],
 
        [[34, 34, 34],
         [34, 34, 34],
         [34, 34, 34],
         ...,
         [22, 22, 22],
         [20, 20, 20],
         [20, 20, 20]],
 
        ...,
 
        [[15, 15, 15],
         [15, 15, 15],
         [15, 15, 15],
         ...,
         [ 6,  6,  6],
         [ 6,  6,  6],
         [ 5,  5,  5]],
 
        [[11, 11, 11],
         [13, 13, 13],
         [13, 13, 13],
         ...,
         [ 6,  6,  6],
         [ 3,  3,  3],
         [ 1,  1,  1]],
 
        [[13, 13, 13],
    

In [ ]:
# @title size올림
model = RTDETR("rtdetr-l.pt")  # 새로 시작 (baseline과 공정 비교 위해 처음부터)
model.train(
    data="/content/cadica_yolo/data.yaml",
    epochs=50,
    imgsz=768,
    batch=4,           # 768로 올리면 T4 메모리 부족 가능 → OOM나면 2로
    lr0=0.0005,
    optimizer="AdamW",
    patience=20,
    amp=True,
    cache=True,
    project="/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result",
    name="train_imgsz768",
)

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_imgsz768, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, ove

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      4.47G      1.404     0.6143     0.4608          1        768: 100% ━━━━━━━━━━━━ 770/770 1.9it/s 6:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 42/42 3.1it/s 13.7s
                   all        335        506    0.00663      0.221     0.0202    0.00301

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      4.47G      1.043     0.6998     0.1542         18        768: 0% ──────────── 0/770  0.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      4.47G     0.9064     0.9434     0.2236          0        768: 100% ━━━━━━━━━━━━ 770/770 2.0it/s 6:22
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 42/42 4.4it/s 9.6s
                   all        335        506     0.0329      0.117     0.0164    0.00369

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      4.47G     0.8072     0.8664     0.1446         11        768: 0% ──────────── 0/770  0.6s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      24/50      4.48G     0.4957     0.4921     0.1195          2        768: 100% ━━━━━━━━━━━━ 770/770 2.0it/s 6:18
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 42/42 4.3it/s 9.8s
                   all        335        506      0.146      0.178     0.0589     0.0178

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      25/50      4.48G     0.4245     0.4158    0.09947          9        768: 0% ──────────── 0/770  0.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      25/50      4.48G     0.4843     0.4854      0.116          0        768: 100% ━━━━━━━━━━━━ 770/770 2.0it/s 6:17
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 42/42 4.3it/s 9.7s
                   all        335        506       0.13      0.217     0.0615     0.0199
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 5, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

25 epochs completed in 2.849 hours.
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_imgsz768/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_imgsz768/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_imgsz76

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78f70585ff20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
BASE = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result"

model = RTDETR(f"{BASE}/train_imgsz768/weights/best.pt")
val  = model.val(data="/content/cadica_yolo/data.yaml", split="val",  imgsz=768)
test = model.val(data="/content/cadica_yolo/data.yaml", split="test", imgsz=768)
print(f"VAL  P={val.box.mp:.3f} R={val.box.mr:.3f} mAP50={val.box.map50:.3f} mAP50-95={val.box.map:.3f}")
print(f"TEST P={test.box.mp:.3f} R={test.box.mr:.3f} mAP50={test.box.map50:.3f} mAP50-95={test.box.map:.3f}")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 54.7±13.0 MB/s, size: 96.0 KB)
val: Scanning /content/cadica_yolo/labels/val.cache... 335 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 335/335 93.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.2it/s 17.9s
                   all        335        506      0.165      0.281      0.102     0.0344
Speed: 3.2ms preprocess, 45.3ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val-3
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 57.3±7.3 MB/s, size: 98.1 KB)
val: Scanning /content/cadica_yolo/

In [ ]:
# @title RCA/LCA 분리학습

# @title 1. LCA/RCA 데이터셋 분리
import os, re
from collections import defaultdict
from pathlib import Path
import pandas as pd

SRC_ROOT = Path("/content/cadica_yolo")
SPLIT_CSV = Path("/content/drive/MyDrive/🐥new_2_project/04_DL_project/08_전처리/common_split.csv")
VIEW_ROOTS = {"LCA": Path("/content/cadica_yolo_lca"), "RCA": Path("/content/cadica_yolo_rca")}
SPLITS = ["train", "val", "test"]
FRAME_PREFIX_RE = re.compile(r"^(p\d+)_(v\d+)_")

def load_view_map(split_csv=SPLIT_CSV):
    df = pd.read_csv(split_csv)
    return {(r.patient_id, r.video_id): r.view for r in df.itertuples()}

def get_view(filename, view_map):
    m = FRAME_PREFIX_RE.match(filename)
    if not m:
        return None
    return view_map.get((m.group(1), m.group(2)))

def build_view_datasets(src_root=SRC_ROOT, split_csv=SPLIT_CSV):
    view_map = load_view_map(split_csv)
    counts = defaultdict(lambda: defaultdict(int))
    unmatched = []

    for split in SPLITS:
        img_dir, lbl_dir = src_root / "images" / split, src_root / "labels" / split
        if not img_dir.exists():
            continue
        for img_path in sorted(img_dir.iterdir()):
            if not img_path.is_file():
                continue
            view = get_view(img_path.name, view_map)
            if view not in VIEW_ROOTS:
                unmatched.append(img_path.name)
                continue
            dst_root = VIEW_ROOTS[view]
            dst_img_dir, dst_lbl_dir = dst_root / "images" / split, dst_root / "labels" / split
            dst_img_dir.mkdir(parents=True, exist_ok=True)
            dst_lbl_dir.mkdir(parents=True, exist_ok=True)

            dst_img = dst_img_dir / img_path.name
            if not dst_img.exists():
                os.symlink(img_path.resolve(), dst_img)

            lbl_path = lbl_dir / f"{img_path.stem}.txt"
            dst_lbl = dst_lbl_dir / lbl_path.name
            if lbl_path.exists() and not dst_lbl.exists():
                os.symlink(lbl_path.resolve(), dst_lbl)

            counts[view][split] += 1

    for view, root in VIEW_ROOTS.items():
        (root / "data.yaml").write_text(
            f"path: {root}\ntrain: images/train\nval: images/val\ntest: images/test\n"
            "nc: 1\nnames: ['lesion']\n"
        )

    print("=== LCA / RCA 분리 결과 (bbox frame 수) ===")
    for view in VIEW_ROOTS:
        row = counts[view]
        print(f"{view}: train={row.get('train',0)} val={row.get('val',0)} "
              f"test={row.get('test',0)}  total={sum(row.values())}")
    if unmatched:
        print(f"[주의] view 매칭 실패 {len(unmatched)}개, 예: {unmatched[:5]}")
    return counts

build_view_datasets()

=== LCA / RCA 분리 결과 (bbox frame 수) ===
LCA: train=1818 val=254 test=153  total=2225
RCA: train=1259 val=81 test=120  total=1460


defaultdict(<function __main__.build_view_datasets.<locals>.<lambda>()>,
            {'LCA': defaultdict(int, {'train': 1818, 'val': 254, 'test': 153}),
             'RCA': defaultdict(int, {'train': 1259, 'val': 81, 'test': 120})})

In [ ]:
# @title 2-1. LCA 학습
from ultralytics import RTDETR

BASE = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result"

model_lca = RTDETR("rtdetr-l.pt")   # 원본 pretrained에서 새로 시작 (통합 학습 편향 방지)
model_lca.train(
    data="/content/cadica_yolo_lca/data.yaml",
    name="train_lca_v1",
    project=BASE,
    epochs=50, imgsz=640, batch=8,
    lr0=0.0005, optimizer="AdamW", amp=True, patience=20,
)

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_yolo_lca/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_lca_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, 

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      6.53G      1.521     0.7824     0.5082         12        640: 100% ━━━━━━━━━━━━ 228/228 1.4it/s 2:45
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.5it/s 10.9s
                   all        254        425    0.00433      0.325    0.00296   0.000515

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      6.53G      1.573     0.4029     0.3778         19        640: 0% ──────────── 0/228  0.9s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      6.53G      1.047     0.7999     0.2711          5        640: 100% ━━━━━━━━━━━━ 228/228 1.5it/s 2:32
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 2.8it/s 5.7s
                   all        254        425      0.024       0.24     0.0202    0.00509

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      6.53G     0.7079      1.266     0.1822          8        640: 0% ──────────── 0/228  1.0s

...

      29/50      6.53G     0.4459     0.4512    0.09682          9        640: 100% ━━━━━━━━━━━━ 228/228 1.5it/s 2:33
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 2.7it/s 6.0s
                   all        254        425      0.223      0.101     0.0557     0.0117

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/50      6.53G     0.4875     0.5232    0.09748         22        640: 0% ──────────── 0/228  1.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/50      6.53G     0.4444     0.4535    0.09768          4        640: 100% ━━━━━━━━━━━━ 228/228 1.5it/s 2:32
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 3.0it/s 5.3s
                   all        254        425      0.253      0.122     0.0805     0.0229

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      31/50      6.53G     0.5518     0.5167     0.1461         23        640: 0% ──────────── 0/228  1.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      31/50      6.53G     0.4387     0.4534    0.09589          3        640: 100% ━━━━━━━━━━━━ 228/228 1.5it/s 2:33
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 2.6it/s 6.0s
                   all        254        425      0.184      0.127     0.0702     0.0162
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 11, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

31 epochs completed in 1.898 hours.
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_lca_v1/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_lca_v1/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_lca_v1/wei

In [ ]:
from ultralytics import RTDETR
BASE = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result"
model = RTDETR(f"{BASE}/train_lca_v1/weights/best.pt")
val  = model.val(data="/content/cadica_yolo_lca/data.yaml", split="val",  imgsz=640)
test = model.val(data="/content/cadica_yolo_lca/data.yaml", split="test", imgsz=640)
print(f"LCA VAL  P={val.box.mp:.3f} R={val.box.mr:.3f} mAP50={val.box.map50:.3f} mAP50-95={val.box.map:.3f}")
print(f"LCA TEST P={test.box.mp:.3f} R={test.box.mr:.3f} mAP50={test.box.map50:.3f} mAP50-95={test.box.map:.3f}")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 61.8±19.0 MB/s, size: 100.2 KB)
val: Scanning /content/cadica_yolo_lca/labels/val.cache... 254 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 254/254 41.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.6it/s 9.9s
                   all        254        425      0.209      0.182     0.0922     0.0404
Speed: 2.2ms preprocess, 32.6ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val-5
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 50.8±9.1 MB/s, size: 101.4 KB)
val: Scanning /content/cadica_

In [ ]:
from ultralytics import RTDETR
BASE = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result"

# 통합 baseline → LCA test
base = RTDETR(f"{BASE}/train_v3_amp_on/weights/best.pt")
base_lca = base.val(data="/content/cadica_yolo_lca/data.yaml", split="test", imgsz=640)
print(f"BASELINE on LCA-TEST  P={base_lca.box.mp:.3f} R={base_lca.box.mr:.3f} "
      f"mAP50={base_lca.box.map50:.3f} mAP50-95={base_lca.box.map:.3f}")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1990.5±471.2 MB/s, size: 106.9 KB)
val: Scanning /content/cadica_yolo_lca/labels/test.cache... 153 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 153/153 42.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.5it/s 6.6s
                   all        153        153      0.739      0.463      0.485      0.242
Speed: 2.8ms preprocess, 34.8ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val-7
BASELINE on LCA-TEST  P=0.739 R=0.463 mAP50=0.485 mAP50-95=0.242


In [ ]:
# @title RCA 학습
from ultralytics import RTDETR

BASE = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result"

model_rca = RTDETR("rtdetr-l.pt")  # 원본 pretrained에서 새로 시작
model_rca.train(
    data="/content/cadica_yolo_rca/data.yaml",
    name="train_rca_v1",
    project=BASE,
    epochs=50,
    imgsz=640,
    batch=8,
    lr0=0.0005,
    optimizer="AdamW",
    amp=True,
    patience=20,
)

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_yolo_rca/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_rca_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, 

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      6.71G      1.384     0.9552     0.4513          9        640: 100% ━━━━━━━━━━━━ 158/158 1.3it/s 1:57
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.0s/it 6.2s
                   all         81         81    0.00214      0.642    0.00412   0.000901

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      6.71G     0.9553     0.9054     0.2473         13        640: 0% ──────────── 0/158  0.9s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      6.72G      0.914     0.9227      0.224          9        640: 100% ━━━━━━━━━━━━ 158/158 1.5it/s 1:44
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.4it/s 1.8s
                   all         81         81     0.0148      0.519     0.0499     0.0119

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      6.72G     0.9855     0.9848     0.3005         14        640: 0% ──────────── 0/158  0.6s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      40/50      6.72G     0.4393     0.4217     0.1023         10        640: 100% ━━━━━━━━━━━━ 158/158 1.5it/s 1:45
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.9it/s 2.0s
                   all         81         81      0.286      0.309      0.188     0.0473
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      41/50      6.72G     0.3875     0.4118     0.1093         13        640: 0% ──────────── 0/158  1.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      41/50      6.72G     0.3897     0.3995     0.1072          6        640: 100% ━━━━━━━━━━━━ 158/158 1.5it/s 1:44
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.4it/s 1.8s
                   all         81         81      0.364      0.148     0.0939     0.0223
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 21, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

41 epochs completed in 1.253 hours.
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_rca_v1/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_rca_v1/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_rca_v1/weigh

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f0a96ca7620>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# @title RCA 평가 + baseline 공정비교
from ultralytics import RTDETR

BASE = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result"

rca = RTDETR(f"{BASE}/train_rca_v1/weights/best.pt")
val  = rca.val(data="/content/cadica_yolo_rca/data.yaml", split="val",  imgsz=640)
test = rca.val(data="/content/cadica_yolo_rca/data.yaml", split="test", imgsz=640)
print(f"RCA VAL  P={val.box.mp:.3f} R={val.box.mr:.3f} mAP50={val.box.map50:.3f} mAP50-95={val.box.map:.3f}")
print(f"RCA TEST P={test.box.mp:.3f} R={test.box.mr:.3f} mAP50={test.box.map50:.3f} mAP50-95={test.box.map:.3f}")

base = RTDETR(f"{BASE}/train_v3_amp_on/weights/best.pt")
base_rca = base.val(data="/content/cadica_yolo_rca/data.yaml", split="test", imgsz=640)
print(f"BASELINE on RCA-TEST P={base_rca.box.mp:.3f} R={base_rca.box.mr:.3f} "
      f"mAP50={base_rca.box.map50:.3f} mAP50-95={base_rca.box.map:.3f}")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2610.9±884.9 MB/s, size: 100.0 KB)
val: Scanning /content/cadica_yolo_rca/labels/val.cache... 81 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 81/81 30.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.4it/s 4.2s
                   all         81         81      0.557      0.259      0.243     0.0644
Speed: 5.1ms preprocess, 39.0ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val-8
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 58.9±11.6 MB/s, size: 106.3 KB)
val: Scanning /content/cadica_y